# Analyse des données

In [2]:
from src.donnees import import_donnees, renomer_cle_jointure, concatenation_annees
from src.nettoyage import recodage, mapping_renommer_colonnes, création_age_usager, jointure

In [3]:
donnees_completes = import_donnees()
donnees_completes["caract"][22].columns

/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')
/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')


Index(['Accident_Id', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long'],
      dtype='object')

Le nom de clé de jointure n'est pas le même que pour tout les autres fichiers, nécessiare de le modifier pour une jointure correct

In [4]:
donnees_completes["caract"][22] = renomer_cle_jointure(donnees_completes["caract"][22], "Num_Acc", "Accident_Id")

# CARACTERISTIQUE
df_caract = concatenation_annees(donnees_completes, "caract")
# LIEUX
df_lieux = concatenation_annees(donnees_completes, "lieux")
# VEHICULE
df_vehicule = concatenation_annees(donnees_completes, "vehicule")
# USAGER
df_usager = concatenation_annees(donnees_completes, "usager")

In [5]:
mappings = mapping_renommer_colonnes()

In [6]:
# recodage des noms des colonnes 
df_caract_recoder = recodage(df_caract, mappings["caract"])
df_lieux_recoder = recodage(df_lieux, mappings["lieux"])
df_vehicule_recoder = recodage(df_vehicule, mappings["vehicule"])
df_usager_recoder = recodage(df_usager, mappings["usager"])

In [7]:
df_lieux_recoder[df_lieux_recoder["Num_Acc"]==202400000005]

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
6,202400000005,Autoroute,BEDOULE (CHEMIN DE LA),0,NaN,Bidirectionnelle,4,Sans objet,Plat,1,0,Partie rectiligne,NaN,17,Mouillée,Aucun,Sur chaussée,50
7,202400000005,Autoroute,BEDOULE (CHEMIN DE LA),0,NaN,Bidirectionnelle,4,Piste cyclable,Pente,272,0,Partie rectiligne,NaN,17,Mouillée,Aucun,Sur chaussée,70


On observe que certains accidents sont en double dans les données, nous pensons qu'il s'agit d'une correction. Par exemple ci-dessus, la deuxième ligne remplace le champ vosp, qui était sans objet, il devient piste cyclabe. En réalisant un merge sans modification, on obtient : des doublons d’accidents, qui créent ensuite des doublons de véhicules, qui créent ensuite des doublons d’usagers. Donc il est nécessaire de garder qu'une seule ligne des numéro d'accident du df lieux.

In [8]:
# supprimer les doublons de corrections des données dans le fichier lieux 
df_lieux_recoder = df_lieux_recoder.drop_duplicates(subset="Num_Acc", keep="last")

# age pour les usagers
df_usager_recoder = création_age_usager(df_usager_recoder)

In [9]:
df_final = jointure(df_caract_recoder, df_lieux_recoder, df_vehicule_recoder, df_usager_recoder)

In [10]:
df_final.columns

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long', 'catr', 'voie', 'v1', 'v2',
       'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout',
       'surf', 'infra', 'situ', 'vma', 'id_vehicule', 'num_veh_x', 'senc',
       'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc', 'id_usager',
       'num_veh_y', 'place', 'catu', 'grav', 'sexe', 'trajet', 'secu1',
       'secu2', 'secu3', 'locp', 'actp', 'etatp', 'age'],
      dtype='object')

In [11]:
df_final

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,grav,sexe,trajet,secu1,secu2,secu3,locp,actp,etatp,age
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Blessé hospitalisé,Homme,Domicile - École,Ceinture,NaN,NaN,-1.0,-1,-1.0,23
1,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,70285,Hors agglomération,Hors intersection,...,Indemne,Homme,Utilisation professionnelle,Ceinture,NaN,NaN,-1.0,-1,-1.0,29
2,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,NaN,NaN,3.0,3,1.0,99
3,202400000002,20,mars,2024,15:05,Plein jour,21,21054,En agglomération,Intersection en T,...,Indemne,Homme,Utilisation professionnelle,Ceinture,Aucun équipement,NaN,3.0,3,1.0,39
4,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,15012,Hors agglomération,Hors intersection,...,Blessé léger,Femme,Promenade - loisirs,Non déterminable,Aucun équipement,NaN,-1.0,-1,-1.0,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377697,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Indemne,Femme,Promenade - loisirs,Ceinture,NaN,NaN,0.0,0,-1.0,24
377698,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Blessé hospitalisé,Femme,Promenade - loisirs,Ceinture,NaN,NaN,0.0,0,-1.0,22
377699,202200055301,1,janvier,2022,08:40,Plein jour,81,81099,Hors agglomération,Intersection en T,...,Blessé léger,Femme,Promenade - loisirs,Ceinture,NaN,NaN,0.0,0,-1.0,73
377700,202200055302,1,mars,2022,16:55,Plein jour,41,41018,En agglomération,Hors intersection,...,Blessé hospitalisé,Homme,Domicile - Travail,Casque,Gants (2RM/3RM),NaN,-1.0,-1,-1.0,34


In [12]:
df_final["id_usager"].nunique()
# le nombre de usager devrait etre le meme nombre que le nb de lignes dans df_final car nous avons plusieurs usagers pour un accidents donc une ligne par usager

377638

In [13]:
df_final[df_final.duplicated(["Num_Acc", "id_vehicule", "id_usager"], keep=False)]
# pas de doublons ! 

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,grav,sexe,trajet,secu1,secu2,secu3,locp,actp,etatp,age


In [14]:
df_final[df_final.duplicated("id_usager", keep=False)]
# Présence de 64 id_usager en NA 

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,grav,sexe,trajet,secu1,secu2,secu3,locp,actp,etatp,age
10,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1111,202400000466,30,avril,2024,19:20,Plein jour,79,79191,En agglomération,Intersection à plus de 4 branches,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
6665,202400002915,20,décembre,2024,16:00,Plein jour,62,62758,En agglomération,Intersection en T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
11558,202400005069,5,juin,2024,11:44,Plein jour,2B,2B033,En agglomération,Hors intersection,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
15975,202400007018,3,juin,2024,14:50,Plein jour,49,49007,En agglomération,Hors intersection,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227810,202300044719,1,octobre,2023,16:35,Plein jour,60,60057,En agglomération,Hors intersection,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
229837,202300045599,11,mai,2023,10:02,Plein jour,75,75117,En agglomération,Intersection en X,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
240635,202300050308,9,mai,2023,20:40,Plein jour,14,14333,En agglomération,Hors intersection,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
242169,202300050996,4,octobre,2023,06:27,Nuit sans éclairage public,68,68286,Hors agglomération,Intersection en T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [15]:
df_final[df_final["Num_Acc"]==202400000004]

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,grav,sexe,trajet,secu1,secu2,secu3,locp,actp,etatp,age
9,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,Blessé léger,Homme,Promenade - loisirs,Casque,Gants (2RM/3RM),NaN,0.0,0,-1.0,63
10,202400000004,24,mars,2024,17:50,Plein jour,14,14118,En agglomération,Intersection en T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [16]:
df_usager[df_usager["Num_Acc"]==202400000004]

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
9,202400000004,203 988 573,155 781 754,B01,1,1,4,1,1963.0,5,2,6,-1,0,0,-1


In [17]:
df_vehicule[df_vehicule["Num_Acc"]==202400000004]

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN
5,202400000004,155 781 755,A01,3,7,0,2,3,15,0,NaN


PB : les usagers de certains vehicules ne sont pas repertorié dans les données.  Nous nous interressons aux usagers car nous regardons notamment les blessures des usagers, sans les données sur eux ce n'est pas possible de faire les analyses. Ces usagers représente 64 sur 377702 usagers, nous pouvons donc les supprimés sans nuire à nos analyses, car ils sont peu nombreux.

In [18]:
print(df_final["Num_Acc"].isna().sum())
print(df_final["id_vehicule"].isna().sum())
print(df_final["id_usager"].isna().sum())   # nan pour les usager seulement à supprimer !!!


0
0
64


In [19]:
df_final = df_final.dropna(subset=["id_usager"])

## Vérification des NA

In [20]:
import pandas as pd
na_table = pd.DataFrame({
    "nb_na": df_final.isna().sum(),
    "pct_na": (df_final.isna().sum() / len(df_final)) * 100
})

print(na_table)


              nb_na     pct_na
Num_Acc           0   0.000000
jour              0   0.000000
mois              0   0.000000
an                0   0.000000
hrmn              0   0.000000
lum               7   0.001854
dep               0   0.000000
com               0   0.000000
agg               0   0.000000
int              28   0.007415
atm               9   0.002383
col             167   0.044222
adr           11886   3.147459
lat               0   0.000000
long              0   0.000000
catr              0   0.000000
voie          55408  14.672252
v1                0   0.000000
v2           344076  91.112653
circ          23501   6.223156
nbv               0   0.000000
vosp          12785   3.385517
prof            523   0.138492
pr                0   0.000000
pr1               0   0.000000
plan            405   0.107246
lartpc       377426  99.943862
larrout           0   0.000000
surf            493   0.130548
infra          4080   1.080400
situ            453   0.119956
vma     

In [21]:
sexe_na = df_final[df_final["sexe"].isna()]        # Seulemetn indemne 
age_na = df_final[df_final["age"].isna()]          # tout sauf Tué
secu1_na = df_final[df_final["secu1"].isna()]      # tout 40 Tué sur environ 10300

circ_na = df_final[df_final["circ"].isna()]        # tout 403 Tué 
infra_na = df_final[df_final["infra"].isna()]      # tout 55 Tué
manv_na = df_final[df_final["manv"].isna()]        # tout 6 Tué
trajet_na = df_final[df_final["trajet"].isna()]    # tout 2216 Tué


In [22]:
trajet_na["grav"].unique()

array(['Blessé léger', 'Indemne', 'Blessé hospitalisé', 'Tué', nan],
      dtype=object)

In [23]:
trajet_na.groupby("grav").size()

grav
Blessé hospitalisé    13309
Blessé léger          44982
Indemne               48198
Tué                    2216
dtype: int64

In [24]:
df_final.groupby("grav").size()

grav
Blessé hospitalisé     57657
Blessé léger          149293
Indemne               159949
Tué                    10380
dtype: int64

Supprimer les na de ces variables : lum, int, atm, col, prof, plan, surf, situ, catv, obs, obsm, choc, sexe (2%), age (2%), secu1 (2%)
Supprimer ces variables : vops, pr, pr1, lartpc, larrout, secu2 (44%), secu3 (96%)  -> elles sont pas intéressantes ou contiennent trop de NA 

Les garder sans supprimer les NA (car trop de tué mais pas utiliser pour le modèle ou faire attention) : circ (6%), infra (1%), manv (0.03%), trajet (29%)

In [25]:
# suppression des NA 
df_final = df_final.dropna(subset=["lum", "int", "atm", "col", "prof", "plan", "surf", "situ", "catv", "obs", "obsm", "choc", "sexe", "age", "secu1"])

In [26]:
# valeur aberrante nbv, age, vma
df_final["age"].unique()
df_final["vma"].unique()

def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR
    
    return df[(df[col] < borne_inf) | (df[col] > borne_sup)]


In [27]:
outliers_age = detect_outliers_iqr(df_final, "age")
outliers_age["age"].unique()


<IntegerArray>
[99, 100, 103, 102, 111, 101, 112, 104, 105, 109, 106, 107, 113]
Length: 13, dtype: Int64

Certaines valeurs comme 113 ans sont un peu bizarres mais acceptable

In [33]:
outliers_vma = detect_outliers_iqr(df_final, "vma")
outliers_vma["vma"].unique()
# beaucoup de valeurs aberrantes

array([ -1, 130,   1,   3, 500, 300,   2, 800, 140,   4, 700,   0, 900])

In [34]:
outliers_vma.groupby("grav").size()

grav
Blessé hospitalisé     4909
Blessé léger           8293
Indemne               11094
Tué                     898
dtype: int64

Des valeurs aberrantes comme 900 ou 4 ! De plus, il y a des gravités "tué" donc pas trop possible de supprimer les valeurs aberrante pour le modèle - suppression de la colonne à faire

In [ ]:
df_final["nbv"].unique() # a ne pas utiliser ! Supprimer la colonne

array(['2', '1', '4', '3', ' -1', '0', '6', '8', '5', '9', '7',
       '#VALEURMULTI', '10', '12', '11', '#ERREUR', 2, 3, 4, 1, 5, 0, 6,
       -1, 8, 10, 9, 7, 12, 11], dtype=object)

In [31]:
# verif les types 
df_final.dtypes


Num_Acc          Int64
jour             int64
mois            object
an               int64
hrmn            object
lum             object
dep             object
com             object
agg             object
int             object
atm             object
col             object
adr             object
lat             object
long            object
catr            object
voie            object
v1               int64
v2              object
circ            object
nbv             object
vosp            object
prof            object
pr              object
pr1             object
plan            object
lartpc          object
larrout         object
surf            object
infra           object
situ            object
vma              int64
id_vehicule     object
num_veh_x       object
senc             int64
catv            object
obs             object
obsm            object
choc            object
manv            object
motor            int64
occutc         float64
id_usager       object
num_veh_y  

In [32]:
# Vérification des véhicules
df_final.groupby("catv").size()

catv
3RM                    2235
Autobus                3972
Autocar                1936
Autre véhicule         1449
Bicyclette            16112
Cyclomoteur           11007
EDP sans moteur         738
EDP à moteur           8184
Engin spécial           237
Indéterminable          435
Motocyclette          30973
PL                     6751
Quad                    801
Scooter               17787
Tracteur agricole       859
Tracteur routier       1880
Train                   168
Tramway                 590
VAE                    2440
VL                   230565
VU                    26567
Voiturette             2013
dtype: int64